In [35]:
import csv
import math
import re
from pathlib import Path
import numpy as np
import pandas as pd

In [46]:
DATA_PATH = Path("data/electronics_sales.csv")
OUTPUT_PATH = Path("output") / "sales_report.txt"


def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = reader.fieldnames or []

    print(f"Rows: {len(rows)}")
    print(f"Columns: {columns}")
    return rows

read_sales()


Rows: 50
Columns: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3004',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'Xiaomi Redmi Buds',
  'quantity': '2',
  'unit_price_kes': '3199',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3005',
  'sale_date': '2026-09-03',
  'store': 'Nakuru Westside',
  'product': 'Tecno Spark 20',
  'quantity': '2',
  'unit_price_kes': '13499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3006',
  'sale_date': '2026-09-03',
  's

In [37]:
def parse_number(value: str | None) -> float | None:
    """Convert a messy numeric string to a float, or return None if invalid."""
    if value is None:
        return None

    cleaned = str(value).strip().upper()
    if cleaned.startswith("KES"):
        cleaned = cleaned[3:].strip()

    word_to_num = {
        "ZERO": 0,
        "ONE": 1,
        "TWO": 2,
        "THREE": 3,
        "FOUR": 4,
        "FIVE": 5,
        "SIX": 6,
        "SEVEN": 7,
        "EIGHT": 8,
        "NINE": 9,
        "TEN": 10
    }

    if cleaned in ("", "N/A", "NONE", "NULL", "-"):
        return None

    if cleaned in word_to_num:
        return word_to_num[cleaned]


    if "," in cleaned:
        if not re.fullmatch(r"[+-]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", cleaned):
            return None
        cleaned = cleaned.replace(",", "")

    try:
        number = float(cleaned)
    except ValueError:
        return None

    return number if math.isfinite(number) else None


print(parse_number("two"))  # Example result: 2


2


In [38]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return clean rows and skipped rows with reasons."""
    clean_rows = []
    skipped_rows = []

    def find_value(row: dict[str, str], *names: str) -> str | None:
        normalized = {
            key.lower().replace("_", "").replace(" ", ""): value
            for key, value in row.items()
        }

        for name in names:
            target = name.lower().replace("_", "").replace(" ", "")
            if target in normalized:
                return normalized[target]
            for key, value in normalized.items():
                if target in key:
                    return value
        return None

    for row_number, row in enumerate(rows, start=1):
        quantity_raw = find_value(row, "quantity", "qty")
        unit_price_raw = find_value(row, "unit_price", "unit price", "price")

        quantity = parse_number(quantity_raw)
        unit_price = parse_number(unit_price_raw)

        if quantity is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid quantity", "data": row}
            )
            continue

        if unit_price is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid unit price", "data": row}
            )
            continue

        revenue = quantity * unit_price

        clean_row = dict(row)
        clean_row["quantity"] = quantity
        clean_row["unit_price"] = unit_price
        clean_row["revenue"] = revenue
        clean_rows.append(clean_row)

    return clean_rows, skipped_rows

In [39]:

def total_revenue(rows: list[dict]) -> float:
    """Calculate total revenue as quantity × unit price."""
    total = 0.0

    for row in rows:
        quantity = row.get("quantity")
        unit_price = row.get("unit_price", row.get("unit_price_kes"))

        if not isinstance(quantity, (int, float)):
            quantity = parse_number(str(quantity))
        if not isinstance(unit_price, (int, float)):
            unit_price = parse_number(str(unit_price))

        if quantity is not None and unit_price is not None:
            total += quantity * unit_price

    return total


raw_rows = read_sales()
clean_rows, skipped_rows = convert_rows(raw_rows)
print(f"Total Revenue: KES {total_revenue(clean_rows):,.2f}")

Rows: 50
Columns: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Total Revenue: KES 1,317,210.00


In [40]:


def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Calculate total revenue grouped by a field."""
    totals = {}

    for row in rows:
        key = row.get(field)
        if key is None:
            continue

        totals[key] = totals.get(key, 0.0) + row.get("revenue", 0.0)

    return totals

# Calculate revenue totals for each requested grouping.
product_totals = revenue_by(clean_rows, "product")
store_totals = revenue_by(clean_rows, "store")
payment_totals = revenue_by(clean_rows, "payment_method")

print("Product revenue breakdown:")
for product, revenue in sorted(
    product_totals.items(), key=lambda item: item[1], reverse=True
):
    print(f"{product}  KES {revenue:,.2f}")

print("\nRevenue by store:")
for store, revenue in sorted(store_totals.items()):
    print(f"{store}  KES {revenue:,.2f}")

print("\nRevenue by payment method:")
for method, revenue in sorted(payment_totals.items()):
    print(f"{method}  KES {revenue:,.2f}")

Product revenue breakdown:
HP 15 Laptop  KES 389,994.00
Hisense 43in Smart TV  KES 272,993.00
Lenovo Tab M10  KES 199,992.00
Tecno Spark 20  KES 161,988.00
Infinix Hot 40  KES 63,996.00
Samsung Galaxy A15  KES 56,997.00
JBL Go 3 Speaker  KES 49,491.00
Anker Power Bank 20000  KES 38,691.00
Xiaomi Redmi Buds  KES 31,990.00
Oraimo FreePods  KES 19,593.00
USB-C Charger 25W  KES 19,487.00
Safaricom Neon Ray  KES 11,998.00

Revenue by store:
Kisumu Mega Plaza  KES 449,678.00
Mombasa City Mall  KES 154,082.00
Nairobi CBD  KES 337,483.00
Nakuru Westside  KES 139,988.00
Westgate  KES 235,979.00

Revenue by payment method:
Card  KES 267,869.00
Cash  KES 310,773.00
M-Pesa  KES 738,568.00


In [41]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the n highest (key, value) pairs, highest first."""
    # Apply a primary revenue sort and a secondary name sort.
    # 1. Sort from highest revenue to lowest.
    # 2. Break equal-revenue ties by product name, alphabetically.
    sorted_items = sorted(
        totals.items(), key=lambda item: (-item[1], item[0].lower())
    )

    return sorted_items[:n]


# Rank products by their calculated revenue.
# Use the product totals calculated above.
top_three = top_n(product_totals, n=3)

print("TOP 3 HIGHEST REVENUE PRODUCTS")
for rank, (product, revenue) in enumerate(top_three, start=1):
    print(f"{rank}. {product} — KES {revenue:,.2f}")



TOP 3 HIGHEST REVENUE PRODUCTS
1. HP 15 Laptop — KES 389,994.00
2. Hisense 43in Smart TV — KES 272,993.00
3. Lenovo Tab M10 — KES 199,992.00


In [42]:
def high_value(rows: list[dict], threshold: float) -> list[dict]:
    """Transactions whose revenue is at or above threshold (use a comprehension)."""
    return [row for row in rows if row.get("revenue", 0.0) >= threshold]


# Calculate high-value sales and counts by store.
# Use the cleaned rows produced earlier.
THRESHOLD_KES = 50000.0

# Filter transactions with the list comprehension.
high_value_sales = high_value(clean_rows, THRESHOLD_KES)

print(
    f"--- HIGH-VALUE TRANSACTIONS (>= KES {THRESHOLD_KES:,.2f}) ---"
)
for sale in high_value_sales:
    print(
        f"ID: {sale['sale_id']} | Store: {sale['store']} | Product: {sale['product']} | Revenue: KES {sale['revenue']:,.2f}"
    )

# Count qualifying transactions by store with a dictionary comprehension.
# Include each store represented in the qualifying transactions.
store_high_value_counts = {
    store: sum(1 for sale in high_value_sales if sale["store"] == store)
    for store in sorted({sale["store"] for sale in raw_rows})
}

print("\n--- HIGH-VALUE TRANSACTION COUNTS BY STORE ---")
for store, count in sorted(
    store_high_value_counts.items()
):
    print(f"Store: {store}: {count} transaction(s)")

--- HIGH-VALUE TRANSACTIONS (>= KES 50,000.00) ---
ID: S-3013 | Store: Westgate | Product: Lenovo Tab M10 | Revenue: KES 99,996.00
ID: S-3017 | Store: Kisumu Mega Plaza | Product: Hisense 43in Smart TV | Revenue: KES 155,996.00
ID: S-3022 | Store: Nairobi CBD | Product: HP 15 Laptop | Revenue: KES 259,996.00
ID: S-3036 | Store: Nakuru Westside | Product: Tecno Spark 20 | Revenue: KES 53,996.00
ID: S-3048 | Store: Kisumu Mega Plaza | Product: HP 15 Laptop | Revenue: KES 129,998.00

--- HIGH-VALUE TRANSACTION COUNTS BY STORE ---
Store: Kisumu Mega Plaza: 2 transaction(s)
Store: Mombasa City Mall: 0 transaction(s)
Store: Nairobi CBD: 1 transaction(s)
Store: Nakuru Westside: 1 transaction(s)
Store: Westgate: 1 transaction(s)


In [43]:
# Reusable statistics implemented without the statistics module.
def mean(values: list[float]) -> float:
    """Calculate the arithmetic mean of a list of numbers."""
    if not values:
        raise ValueError("The list of values is empty.")
    return sum(values) / len(values)



def median(values: list[float]) -> float:
    """Calculate the median value of a list of numbers."""
    if not values:
        raise ValueError("The list of values is empty.")

    
    sorted_values = sorted(values)
    n = len(sorted_values)
    mid_index = n // 2

    if n % 2 == 0:  # Average the two middle values when the count is even.
        return (sorted_values[mid_index - 1] + sorted_values[mid_index]) / 2
    else:  # Return the middle value when the count is odd.
        return sorted_values[mid_index]



def minimum_maximum(values: list[float]) -> tuple[float, float]:
    """Find the minimum and maximum values without using min() or max() built-ins."""

    if not values:
        raise ValueError("The list of values is empty.")

    min_value = float('inf')
    max_value = float('-inf')

    for value in values:
        if value < min_value:
            min_value = value
        if value > max_value:
            max_value = value

    return min_value, max_value

# Example: mean([1, 2, 3, 4, 5, 6, 7, 8, 9]) is 5.0.
# Example: median([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) is 5.5.
# Example: minimum_maximum([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) is (1, 10).

# Use the cleaned rows produced earlier.
# Collect transaction revenue using a list comprehension.
revenue_stream = [row["revenue"] for row in clean_rows]

# Calculate the requested revenue statistics.
revenue_mean = mean(revenue_stream)
revenue_median = median(revenue_stream)
revenue_min, revenue_max = minimum_maximum(revenue_stream)


# Print the calculated statistics.
print("--- PER-TRANSACTION REVENUE STATISTICS ---")
print(f"Mean Transaction Value   : KES {revenue_mean:,.2f}")
print(f"Median Transaction Value : KES {revenue_median:,.2f}")
print(f"Minimum Transaction Value: KES {revenue_min:,.2f}")
print(f"Maximum Transaction Value: KES {revenue_max:,.2f}")


--- PER-TRANSACTION REVENUE STATISTICS ---
Mean Transaction Value   : KES 28,025.74
Median Transaction Value : KES 10,998.00
Minimum Transaction Value: KES 1,499.00
Maximum Transaction Value: KES 259,996.00


In [44]:
def verify_with_pandas(path: Path, my_results: dict) -> None:
    """Recompute the headline numbers with pandas and print whether each one matches."""
    # Load the CSV into a pandas DataFrame.
    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    # Apply the same parser used by the custom analysis.
    df["qty_clean"] = df["quantity"].apply(parse_number)
    price_col = [col for col in df.columns if "price" in col.lower()][0]
    df["price_clean"] = df[price_col].apply(parse_number)

    # Exclude rows whose quantity or unit price could not be parsed.
    df_clean = df.dropna(subset=["qty_clean", "price_clean"]).copy()
    df_clean["revenue"] = df_clean["qty_clean"] * df_clean["price_clean"]

    # Calculate the pandas reference metrics.
    pandas_results = {
        "total_revenue": float(df_clean["revenue"].sum()),
        "mean": float(df_clean["revenue"].mean()),
        "median": float(df_clean["revenue"].median()),
        "min": float(df_clean["revenue"].min()),
        "max": float(df_clean["revenue"].max()),
    }

    # Compare each metric using a small floating-point tolerance.
    print("--- PANDAS AUTOMATED AUDIT METRICS ---")
    for metric, pd_val in pandas_results.items():
        my_val = my_results.get(metric)

        if my_val is None:
            status = "MISMATCH (Metric missing from custom results)"
        elif np.isclose(pd_val, my_val, atol=0.01):
            status = "MATCH"
        else:
            status = f"MISMATCH (Pandas: {pd_val:,.2f} vs Custom: {my_val:,.2f})"

        print(f"{metric.upper().replace('_', ' '):<15} : {status}")

In [45]:
if __name__ == "__main__":
    # Read the source sales records.
    path = DATA_PATH
    raw_rows = read_sales(path)

    # Parse numeric values and retain rows that can be cleaned.
    clean_rows, skipped_rows = convert_rows(raw_rows)

    # Calculate the custom revenue aggregates.
    revenue_stream = [row["revenue"] for row in clean_rows]
    total_rev = sum(revenue_stream)
    product_totals = revenue_by(clean_rows, "product")
    top_three = top_n(product_totals, n=3)
    high_value_sales = high_value(clean_rows, threshold=50000.0)

    rev_mean = mean(revenue_stream)
    rev_median = median(revenue_stream)
    rev_min, rev_max = minimum_maximum(revenue_stream)

    # Print the sales report.
    print(f"=== Sales report: September 2026 ===")
    print(f"Rows read          : {len(raw_rows)}")
    print(f"Rows used          : {len(clean_rows)}")

    skipped_details = ", ".join(
        [f"{s['data'].get('sale_id', 'Unknown')}: {s['reason']}" for s in skipped_rows]
    )
    print(f"Rows skipped       : {len(skipped_rows)}   ({skipped_details})")
    print(f"\nTotal revenue (KES): {total_rev:,.2f}")

    print(f"\nRevenue by product (KES)")
    sorted_products = sorted(
        product_totals.items(), key=lambda x: x[1], reverse=True
    )
    for product, total in sorted_products:
        print(f"  {product:<25} {total:,.2f}")

    print(f"\nTop 3 products")
    for idx, (product, total) in enumerate(top_three, start=1):
        print(f"  {idx}. {product} - KES {total:,.2f}")

    print(f"\nHigh-value transactions (>= KES 50,000.00): {len(high_value_sales)}")
    for sale in high_value_sales:
        print(
            f"  {sale['sale_id']}  {sale['sale_date']}  {sale['store']:<18}  {sale['product']:<22}  KES {sale['revenue']:,.2f}"
        )

    print(f"\nTransaction revenue statistics")
    print(f"  mean   : {rev_mean:,.2f}")
    print(f"  median : {rev_median:,.2f}")
    print(f"  min    : {rev_min:,.2f}")
    print(f"  max    : {rev_max:,.2f}\n")

    # Recalculate the report metrics with pandas for verification.
    verify_with_pandas(
        path,
        {
            "total_revenue": total_rev,
            "mean": rev_mean,
            "median": rev_median,
            "min": rev_min,
            "max": rev_max,
        },
    )


Rows: 50
Columns: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
=== Sales report: September 2026 ===
Rows read          : 50
Rows used          : 47
Rows skipped       : 3   (S-3030: Invalid unit price, S-3035: Invalid quantity, S-3039: Invalid unit price)

Total revenue (KES): 1,317,210.00

Revenue by product (KES)
  HP 15 Laptop              389,994.00
  Hisense 43in Smart TV     272,993.00
  Lenovo Tab M10            199,992.00
  Tecno Spark 20            161,988.00
  Infinix Hot 40            63,996.00
  Samsung Galaxy A15        56,997.00
  JBL Go 3 Speaker          49,491.00
  Anker Power Bank 20000    38,691.00
  Xiaomi Redmi Buds         31,990.00
  Oraimo FreePods           19,593.00
  USB-C Charger 25W         19,487.00
  Safaricom Neon Ray        11,998.00

Top 3 products
  1. HP 15 Laptop - KES 389,994.00
  2. Hisense 43in Smart TV - KES 272,993.00
  3. Lenovo Tab M10 - KES 199,992.00

High-value transactions (>= KES 50,000.00)